# Network Expansion
## Model 1 (Test) - Deterministic Baseline

Single-period MILP with fixed demand and budget (perfect foresight). No intertemporal updates or demand growth.

### 1 - Imports

In [1]:
import numpy as np
import pandas as pd

from src.classes import DistributionNetwork, Substation
from src.solver import solve_network, print_results

### 2 - Define the Distribution Network (fixed inputs)

In [2]:
# Nodes and loads
NODES = [f"N{i}" for i in range(1, 14)]
LOADS = [f"D{i}" for i in range(1, 11)]

# Initial substation
S1 = Substation("S1", "N4", 40, ['N3', 'N5', 'N9'], r_cost=200, edge_cost=50)
SUBSTATIONS = [S1]

line_cost = 50

load_capacity = {
    'D1': 6, 'D2': 3, 'D3': 2, 'D4': 5, 'D5': 3,
    'D6': 2, 'D7': 3, 'D8': 5, 'D9': 4, 'D10': 6
}

loads_locations = {
    'D1': 'N1', 'D2': 'N2', 'D3': 'N3', 'D4': 'N6', 'D5': 'N7',
    'D6': 'N8', 'D7': 'N9', 'D8': 'N11', 'D9': 'N12', 'D10': 'N13'
}

nodes_connected = {
    'N1': ['N2'],
    'N2': ['N1', 'N3'],
    'N3': ['N2', 'N4'],
    'N4': ['N3', 'N5', 'N9'],
    'N5': ['N4', 'N6'],
    'N6': ['N5','N7', 'N8'],
    'N7': ['N6'],
    'N8': ['N6'],
    'N9': ['N4', 'N10'],
    'N10': ['N9', 'N11', 'N13'],
    'N11': ['N10', 'N12'],
    'N12': ['N11'],
    'N13': ['N10']
}

DistributionNetwork = DistributionNetwork(
    NODES.copy(),
    LOADS.copy(),
    SUBSTATIONS.copy(),
    load_capacity.copy(),
    nodes_connected.copy(),
    loads_locations.copy(),
    line_cost
)

# Candidate substations (same as earlier models)
capacity = 15
s_cost = 100       # activation cost
l_cost = line_cost # feeder line cost
r_cost = 200       # capacity reinforcement cost

S2 = Substation("S2", "N14", capacity, ['N2'], r_cost, edge_cost=l_cost, fix_cost=s_cost)
S3 = Substation("S3", "N15", capacity, ['N6'], r_cost, edge_cost=l_cost, fix_cost=s_cost)
S4 = Substation("S4", "N16", capacity, ['N11', 'N13'], r_cost, edge_cost=l_cost, fix_cost=s_cost)

DistributionNetwork.add_candidate_substations([S2, S3, S4])

### 3 - Solve single-period deterministic model

In [5]:
R = 10    # size of a single capacity reinforcement
B = 550   # single-period budget (nominal)

# Fixed demand; no growth or state updates
system_demand = sum(DistributionNetwork.load_capacity.values())

solution = solve_network(DistributionNetwork, R, B, OutputFlag=0)
print_results(DistributionNetwork, solution, detailed=True)

System demand: 39
Total cost (nominal): 0.0

Substation activation and supply:
S1 at N4: w=1.0, r=39.00
S2 at N14: w=0.0, r=0.00
S3 at N15: w=0.0, r=0.00
S4 at N16: w=0.0, r=0.00

Node assignments:
Node N1 assigned to S1
Node N2 assigned to S1
Node N3 assigned to S1
Node N4 assigned to S1
Node N5 assigned to S1
Node N6 assigned to S1
Node N7 assigned to S1
Node N8 assigned to S1
Node N9 assigned to S1
Node N10 assigned to S1
Node N11 assigned to S1
Node N12 assigned to S1
Node N13 assigned to S1

Arcs used:
Arc N2 -> N1 assigned to S1
Arc N3 -> N2 assigned to S1
Arc N4 -> N3 assigned to S1
Arc N4 -> N5 assigned to S1
Arc N4 -> N9 assigned to S1
Arc N5 -> N6 assigned to S1
Arc N6 -> N7 assigned to S1
Arc N6 -> N8 assigned to S1
Arc N9 -> N10 assigned to S1
Arc N10 -> N11 assigned to S1
Arc N10 -> N13 assigned to S1
Arc N11 -> N12 assigned to S1


### 4 - Results summary

In [4]:
results_df = pd.DataFrame({
    'System Demand': [round(system_demand, 2)],
    'System Supply': [round(sum(solution['r'].values()), 2)],
    'System Capacity': [sum(solution['P'].values())],
    'Substations Active': [[s for s, w in solution['w'].items() if w == 1]],
    'Cost': [round(solution['objective'], 2)]
})

results_df

,System Demand,System Supply,System Capacity,Substations Active,Cost
0,39,39.0,40.0,[1],0.0
